# Kernel Perceptron

This notebook follows the lecture derivation from the ordinary Perceptron in feature space to the kernel Perceptron representation.

## 1. Perceptron in feature space

Whenever example $i$ is misclassified, the feature-space update is

$$\theta \leftarrow \theta + y_i\phi(x_i).$$

Starting from $\theta=0$, the final parameter can therefore be written as

$$\theta=\sum_j\alpha_jy_j\phi(x_j),$$

where $\alpha_j$ counts how many times example $j$ caused an update.

In [1]:
import numpy as np

def phi(X):
    X = np.asarray(X)
    return np.column_stack([X[:, 0], X[:, 1], X[:, 0] * X[:, 1]])

X = np.array([[1, 1], [1, -1], [-1, 1], [-1, -1]], dtype=float)
y = np.array([1, -1, -1, 1])
X_phi = phi(X)

def perceptron_feature_space(X_phi, y, epochs=10):
    theta = np.zeros(X_phi.shape[1])
    for _ in range(epochs):
        for xi, yi in zip(X_phi, y):
            if yi * (theta @ xi) <= 0:
                theta += yi * xi
    return theta

theta = perceptron_feature_space(X_phi, y)
feature_scores = X_phi @ theta

print('Feature-space theta:', theta)
print('Training scores:', feature_scores)
print('Predictions:', np.where(feature_scores >= 0, 1, -1))

Feature-space theta: [0. 0. 2.]
Training scores: [ 2. -2. -2.  2.]
Predictions: [ 1 -1 -1  1]


## 2. Rewrite prediction using inner products

For a new example $x$,

$$\theta^T\phi(x)=\sum_j\alpha_jy_j\phi(x_j)^T\phi(x).$$

Defining $K(x_j,x)=\phi(x_j)^T\phi(x)$ gives

$$\theta^T\phi(x)=\sum_j\alpha_jy_jK(x_j,x).$$

In [2]:
def feature_map_kernel_matrix(A, B):
    A_phi = phi(A)
    B_phi = phi(B)
    return A_phi @ B_phi.T

def kernel_perceptron(X, y, kernel_matrix, epochs=10):
    n = len(X)
    alpha = np.zeros(n, dtype=int)

    for _ in range(epochs):
        for i in range(n):
            score = np.sum(alpha * y * kernel_matrix[:, i])
            if y[i] * score <= 0:
                alpha[i] += 1

    return alpha

K = feature_map_kernel_matrix(X, X)
alpha = kernel_perceptron(X, y, K)
kernel_scores = K.T @ (alpha * y)

print('Alpha:', alpha)
print('Kernel scores:', kernel_scores)
print('Predictions:', np.where(kernel_scores >= 0, 1, -1))

Alpha: [1 0 0 1]
Kernel scores: [ 2. -2. -2.  2.]
Predictions: [ 1 -1 -1  1]


## 3. Compare the two representations

The feature-space and kernel implementations make the same predictions because they use exactly the same inner products, with the kernel computed from the feature map.

In [3]:
print('Feature-space scores:', feature_scores)
print('Kernel scores:       ', kernel_scores)
print('Scores agree:', np.allclose(feature_scores, kernel_scores))
print('Predictions agree:', np.array_equal(
    np.where(feature_scores >= 0, 1, -1),
    np.where(kernel_scores >= 0, 1, -1)
))

assert np.allclose(feature_scores, kernel_scores)

Feature-space scores: [ 2. -2. -2.  2.]
Kernel scores:        [ 2. -2. -2.  2.]
Scores agree: True
Predictions agree: True


## 4. The kernel Perceptron update

Initialize all coefficients to zero:

$$\alpha_j=0.$$

For training example $i$, compute

$$s_i=\sum_j\alpha_jy_jK(x_j,x_i).$$

If $y_is_i\le0$, increment

$$\alpha_i\leftarrow\alpha_i+1.$$

The high-dimensional parameter vector is never explicitly constructed.

## 5. Apply a polynomial kernel

The previous sections used a kernel obtained from an explicit feature map. We can instead specify the kernel directly.

For a polynomial kernel of degree $p$,

$$K_p(x,z)=(1+x^Tz)^p.$$

In `02_kernel_trick.ipynb`, validation was used to select the degree. We keep $p=2$ here as a simple example, and then apply the selected degree $p=3$.

The Kernel Perceptron itself does not change; only the kernel used to compute the inner products changes.

In [4]:
def polynomial_kernel_matrix(A, B, degree):
    return (1 + A @ B.T) ** degree

def kernel_perceptron_polynomial(X, y, degree, epochs=10):
    K = polynomial_kernel_matrix(X, X, degree)
    alpha = kernel_perceptron(X, y, K, epochs=epochs)
    scores = K.T @ (alpha * y)
    predictions = np.where(scores >= 0, 1, -1)
    return alpha, K, scores, predictions

In [5]:
degree = 2
alpha_2, K_2, scores_2, predictions_2 = kernel_perceptron_polynomial(
    X, y, degree=degree
)

print('Polynomial degree:', degree)
print('Alpha:', alpha_2)
print('Kernel scores:', scores_2)
print('Predictions:', predictions_2)

Polynomial degree: 2
Alpha: [1 1 1 1]
Kernel scores: [ 8. -8. -8.  8.]
Predictions: [ 1 -1 -1  1]


### Applying the degree selected by our Kernel Perceptron experiment

`02_kernel_trick.ipynb` uses the Kernel Perceptron to perform the complete model-selection experiment. For this dataset, that experiment selects $p=3$. We now apply that selected degree to the same Kernel Perceptron algorithm.

$$K_3(x,z)=(1+x^Tz)^3.$$

The learning rule is unchanged: the selected kernel simply defines the inner products used by the Perceptron.

`02_kernel_trick.ipynb` also compares this result with scikit-learn's polynomial SVM, which selects $p=4$. We keep $p=4$ only as a separate external-library comparison.

In [6]:
selected_degree = 3
alpha_3, K_3, scores_3, predictions_3 = kernel_perceptron_polynomial(
    X, y, degree=selected_degree
)

print('Polynomial degree:', selected_degree)
print('Polynomial kernel: K(x, z) = (1 + x^T z)^3')
print('Alpha:', alpha_3)
print('Kernel scores:', scores_3)
print('Predictions:', predictions_3)

Polynomial degree: 3
Polynomial kernel: K(x, z) = (1 + x^T z)^3
Alpha: [1 1 1 1]
Kernel scores: [ 24. -24. -24.  24.]
Predictions: [ 1 -1 -1  1]


### Comparing $p=2$, $p=3$, and $p=4$

The Kernel Perceptron update rule does not change. Only the kernel changes:

$$K_2(x,z)=(1+x^Tz)^2$$

$$K_3(x,z)=(1+x^Tz)^3$$

$$K_4(x,z)=(1+x^Tz)^4.$$ 

`02_kernel_trick.ipynb` selected $p=3$ when the Kernel Perceptron itself was used for validation. The $p=4$ example comes from the separate scikit-learn SVM comparison. The two values need not agree because SVM and Kernel Perceptron are different learning algorithms.

A larger degree corresponds to a richer implicit polynomial feature representation.

### Takeaway

The kernel Perceptron is the ordinary Perceptron rewritten so that feature-space inner products are evaluated by a kernel. The kernel degree is a separate model-selection choice: once validation selects $p$, the same Kernel Perceptron can be run with that kernel.

For this experiment, the Kernel Perceptron selected $p=3$. The external SVM comparison selected $p=4$. This illustrates an important model-selection principle: **the best hyperparameter can depend on the learning algorithm used to fit the model.**